In [0]:
%sql
CREATE OR REPLACE TABLE data_warehouse_factory.silver.silver_production_plan AS
SELECT 
  -- Surrogated / Business Key
  md5(concat_ws('||', line_code, cell_code, start_datetime, end_datetime, _source_file)) AS production_plan_key,
  
  -- Wymiary produkcyjne
  line_code,
  line_name,
  cell_code,
  cell_name,
  
  -- Konwersja dat i czasów
  CAST(start_datetime AS TIMESTAMP) AS start_datetime,
  CAST(end_datetime AS TIMESTAMP) AS end_datetime,
  CAST(start_datetime AS DATE) AS plan_date,
  
  -- Metryki i właściwości planu
  plan_status,
  CAST(planned_capacity_pct AS INT) AS planned_capacity_pct,
  
  -- Wyliczenie długości trwania planu w minutach
  ROUND((CAST(end_datetime AS LONG) - CAST(start_datetime AS LONG)) / 60, 2) AS plan_duration_minutes,
  
  remarks,
  source,
  
  -- Metadane lineage
  _source_file,
  _ingested_at AS _bronze_ingested_at,
  current_timestamp() AS _silver_ingested_at
FROM data_warehouse_factory.bronze.bronze_production_plan
WHERE line_code IS NOT NULL AND cell_code IS NOT NULL;